# Stacking cho bài toán review xấu của Olist: có đáng không?

Cùng bài toán, cùng định nghĩa và mốc thời gian với module 10, 11 và bài Intermediate 01. Câu hỏi
lần này không phải "model nào tốt nhất" mà là câu hỏi của người phải vận hành hệ thống:
**kết hợp nhiều model (stacking) có tốt hơn đủ để bù chi phí không?**

Phần code được chấm là bộ khung stacking **theo thời gian**, đúng cách: chia fold mở rộng dần
(expanding window), dự đoán out-of-fold chỉ dùng quá khứ, meta-model học trên logit của OOF. Phần
phân tích trên dữ liệu thật và `REPORT.md` chấm theo rubric.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) là chỗ phân tích dữ liệu thật; bạn tự do sửa/thêm.
- Chấm: `uv run pytest tracks/data-science/modules/12-advanced-ml/exercises/advanced/01-olist-stacking -q`

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score

TRAIN_END = pd.Timestamp("2018-01-01")
VAL_END = pd.Timestamp("2018-04-01")
TEST_END = pd.Timestamp("2018-09-01")

## 0. Dữ liệu (đã viết sẵn, giống bài Intermediate 01)

In [ ]:
CATEGORICAL = ["product_category_name", "seller_state", "customer_state", "payment_type"]
FEATURES = ["n_items", "n_sellers", "n_products", "price", "freight", "weight", "installments",
            "n_payments", "delivery_days", "promised_days", "days_vs_estimate", "is_late",
            "approve_hours", "carrier_days", "freight_ratio", "same_state", "purchase_dow"] + CATEGORICAL


def build_dataset(raw_dir) -> pd.DataFrame:
    """Một dòng mỗi đơn giao xong có review: FEATURES + bad_review + order_purchase_timestamp."""
    from pathlib import Path
    raw = Path(raw_dir)
    dates = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
    orders = pd.read_csv(raw / "olist_orders_dataset.csv", parse_dates=dates)
    reviews = pd.read_csv(raw / "olist_order_reviews_dataset.csv", parse_dates=["review_answer_timestamp"])
    items = pd.read_csv(raw / "olist_order_items_dataset.csv")
    payments = pd.read_csv(raw / "olist_order_payments_dataset.csv")
    customers = pd.read_csv(raw / "olist_customers_dataset.csv")
    sellers = pd.read_csv(raw / "olist_sellers_dataset.csv")
    products = pd.read_csv(raw / "olist_products_dataset.csv")

    df = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
    latest = reviews.sort_values("review_answer_timestamp", kind="stable").drop_duplicates("order_id", keep="last")
    df = df.merge(latest[["order_id", "review_score"]], on="order_id")
    df["bad_review"] = (df.review_score <= 2).astype(int)

    it = (items.merge(products[["product_id", "product_category_name", "product_weight_g"]], on="product_id", how="left")
               .merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left"))
    per_order = it.groupby("order_id").agg(
        n_items=("order_item_id", "size"), n_sellers=("seller_id", "nunique"), n_products=("product_id", "nunique"),
        price=("price", "sum"), freight=("freight_value", "sum"), weight=("product_weight_g", "sum"))
    first_item = (it.sort_values("order_item_id").drop_duplicates("order_id")
                    [["order_id", "product_category_name", "seller_state"]])
    pay = payments.groupby("order_id").agg(installments=("payment_installments", "max"),
                                           n_payments=("payment_sequential", "size"),
                                           payment_type=("payment_type", "first"))
    df = (df.merge(per_order, on="order_id", how="left").merge(first_item, on="order_id", how="left")
            .merge(pay, on="order_id", how="left")
            .merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left"))

    day = pd.Timedelta(days=1)
    df["delivery_days"] = (df.order_delivered_customer_date - df.order_purchase_timestamp) / day
    df["promised_days"] = (df.order_estimated_delivery_date - df.order_purchase_timestamp) / day
    df["days_vs_estimate"] = (df.order_delivered_customer_date - df.order_estimated_delivery_date) / day
    df["is_late"] = (df.order_delivered_customer_date > df.order_estimated_delivery_date).astype(int)
    df["approve_hours"] = (df.order_approved_at - df.order_purchase_timestamp) / pd.Timedelta(hours=1)
    df["carrier_days"] = (df.order_delivered_carrier_date - df.order_purchase_timestamp) / day
    df["freight_ratio"] = df.freight / df.price
    df["same_state"] = (df.customer_state == df.seller_state).astype(int)
    df["purchase_dow"] = df.order_purchase_timestamp.dt.day_of_week
    for col in CATEGORICAL:
        df[col] = df[col].astype("category")
    return df[["order_id", "order_purchase_timestamp"] + FEATURES + ["bad_review"]].reset_index(drop=True)


def time_split(df: pd.DataFrame, date_col: str = "order_purchase_timestamp") -> dict[str, pd.DataFrame]:
    """{"train", "val", "test"} theo ngày mua; bỏ đơn mua từ TEST_END trở đi."""
    t = df[date_col]
    return {"train": df[t < TRAIN_END], "val": df[(t >= TRAIN_END) & (t < VAL_END)],
            "test": df[(t >= VAL_END) & (t < TEST_END)]}

## 1. Fold theo thời gian

`expanding_folds(order, n_folds=4)`: sắp vị trí các dòng theo `order` (thời điểm mua, sort ổn
định), chia thành `n_folds + 1` khối liên tiếp gần bằng nhau (`np.array_split`). Fold k học trên các
khối 0…k và dự đoán khối k + 1. Trả list `(fit_idx, pred_idx)` là **vị trí** (dùng với `iloc`).
Khối đầu tiên không bao giờ được dự đoán.

```
khối:   [0][1][2][3][4]
fold 0:  F  P
fold 1:  F  F  P
fold 2:  F  F  F  P
fold 3:  F  F  F  F  P
```

In [ ]:
def expanding_folds(order, n_folds: int = 4) -> list[tuple[np.ndarray, np.ndarray]]:
    """Fold k: học trên khối 0..k, dự đoán khối k + 1 (khối theo thứ tự thời gian)."""
    # TODO: pos = np.argsort(order, kind="stable"); blocks = np.array_split(pos, n_folds + 1)
    raise NotImplementedError

## 2. Dự đoán out-of-fold

`oof_predictions(models, X, y, folds)`: `models` là dict `tên → hàm không tham số trả về model
chưa fit` (có `fit` và `predict_proba`). Với mỗi fold và mỗi model: tạo model **mới**, fit trên
`fit_idx`, ghi xác suất lớp 1 vào các dòng `pred_idx`. Trả DataFrame cùng index với `X`, mỗi cột
một model, `NaN` ở các dòng không fold nào dự đoán.

In [ ]:
def oof_predictions(models: dict, X: pd.DataFrame, y, folds) -> pd.DataFrame:
    """OOF: mỗi dòng được dự đoán bởi model chỉ học trên các khối trước nó."""
    # TODO
    raise NotImplementedError

## 3. Meta-model

- `to_logit(P, eps=1e-6)`: kẹp xác suất vào `[eps, 1 − eps]` rồi lấy $$\log\frac{p}{1-p}$$.
- `fit_meta(oof, y)`: `LogisticRegression` (mặc định, `max_iter=1000`) trên `to_logit` của các dòng
  OOF **không có NaN**. Meta-model tuyến tính trên logit là lựa chọn mặc định an toàn: ít tham số,
  khó overfit, hệ số đọc được như "tin model nào bao nhiêu".

In [ ]:
def to_logit(P, eps: float = 1e-6) -> np.ndarray:
    """log(p / (1 − p)) sau khi kẹp p vào [eps, 1 − eps]."""
    # TODO
    raise NotImplementedError


def fit_meta(oof: pd.DataFrame, y) -> LogisticRegression:
    """Logistic regression trên logit của OOF, bỏ các dòng có NaN."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: dữ liệu thật. "dev" = train + validation (mua trước 2018-04-01), test giữ nguyên.
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
data = build_dataset(ROOT / "data" / "raw")
splits = time_split(data)
dev = pd.concat([splits["train"], splits["val"]])
test = splits["test"]
print(len(dev), len(test))

In [ ]:
# explore: các model tầng 1. Đây chỉ là điểm xuất phát: thêm, bớt, tune tuỳ bạn.
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = [f for f in FEATURES if f not in CATEGORICAL]


def logistic():
    pre = make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
        (OneHotEncoder(handle_unknown="ignore", min_frequency=50), CATEGORICAL))
    return make_pipeline(pre, LogisticRegression(max_iter=2000, C=0.5))


def lightgbm():
    return lgb.LGBMClassifier(n_estimators=120, learning_rate=0.05, min_child_samples=50, subsample=0.8,
                              subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=1, verbose=-1,
                              random_state=0)


MODELS = {"logistic": logistic, "lightgbm": lightgbm}

In [ ]:
# explore: OOF trên dev, meta-model, rồi đánh giá trên test
folds = expanding_folds(dev.order_purchase_timestamp.to_numpy(), n_folds=4)
oof = oof_predictions(MODELS, dev[FEATURES], dev.bad_review, folds)
meta = fit_meta(oof, dev.bad_review)
print("hệ số meta:", dict(zip(oof.columns, meta.coef_.ravel().round(3).tolist())))
print(oof.dropna().corr().round(3))

fitted = {name: make().fit(dev[FEATURES], dev.bad_review) for name, make in MODELS.items()}
P = pd.DataFrame({name: m.predict_proba(test[FEATURES])[:, 1] for name, m in fitted.items()}, index=test.index)
candidates = {**{name: P[name] for name in P}, "trung bình": P.mean(axis=1),
              "stacking": meta.predict_proba(to_logit(P.to_numpy()))[:, 1]}
pd.DataFrame({name: {"auc": roc_auc_score(test.bad_review, p), "pr_auc": average_precision_score(test.bad_review, p),
                     "logloss": log_loss(test.bad_review, p)} for name, p in candidates.items()}).T.round(4)

## 4. Việc còn lại (tự làm, không chấm tự động)

1. Chênh lệch AUC giữa stacking và model đơn tốt nhất có lớn hơn nhiễu không? Bootstrap tập test.
2. Thêm một model tầng 1 khác bản chất (random forest, kNN trên vài feature, model chỉ dùng
   `days_vs_estimate`). Stacking hưởng lợi từ model **khác** chứ không phải model **mạnh**: xem ma
   trận tương quan OOF.
3. Chi phí: thời gian train, độ trễ khi dự đoán, số thành phần phải giám sát, retrain.
4. Viết `REPORT.md` theo khung có sẵn.